In [1]:
import os

In [2]:
%pwd

'c:\\Users\\Sandeep\\Desktop\\Projects\\AI-book-recommender\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\Sandeep\\Desktop\\Projects\\AI-book-recommender'

In [5]:
# 3 entites update
# from config.ymal
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen = True)
class DataProcessingConfig:
    root_dir: Path
    local_data_file: Path

In [6]:
from Book_recommender.constant import *
from Book_recommender.utils.common import read_yaml, create_directories

In [7]:
class configurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,     # Access to constants
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath) # read all config and params yaml files
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_preprocessing_config(self) -> DataProcessingConfig:
        config = self.config.data_processing

        create_directories([config.root_dir])

        data_processing_config = DataProcessingConfig(
            root_dir = config.root_dir,
            local_data_file = config.local_data_file
        )

        return data_processing_config

In [8]:
import os
import pandas as pd
from Book_recommender.logging.log import logger
import re

In [21]:
class DataProcessing:
    def __init__(self, config):
        self.config = config

    @staticmethod
    def clean_genre(genre_str):
        """ Get the top 1 genre for the book or
        Blank for the null values
        """
        if pd.isna(genre_str) or genre_str in ("Unknown", "Error", ""):
            return ""  # empty string, not "Unknown"
        
        first = str(genre_str).split(",")[0].strip().title()
        return first
    
    @staticmethod
    def clean_text(text):
        '''
        Removes Punctuation
        '''
        text = re.sub(r'[^a-zA-Z0-9\s]', '', str(text))

        return text
           
    def process(self, data):
        # Clean genre
        data["genre_clean"] = data["genre"].apply(DataProcessing.clean_genre)

        # Drop unnecessary column
        data.drop(["bookID",
                   "isbn",
                   "text_reviews_count",
                   "publication_date",
                   "publisher",
                   "Unnamed: 12",
                   "genre"],
                   axis = 1,
                   inplace = True)
        
        # create combined features  
        data["combined_features"] = (
            data["title"].fillna('') + " " +
            data["authors"].fillna('') + " " +
            data["genre_clean"].fillna('') + " " +
            data["description"].fillna(''))
        
        data["average_rating"] = pd.to_numeric(
            data["average_rating"],
            errors="coerce")
        
        # Lowercaseing       
        data["combined_features"] = data["combined_features"].str.lower()

        # Remove punctuation
        data["combined_features"] = data["combined_features"].apply(DataProcessing.clean_text)

        # Save to disk
        data.to_csv(self.config.local_data_file, index=False)

        return data

In [22]:
try:
    config = configurationManager()
    data_preprocessing_config = config.get_data_preprocessing_config()
    
    data = pd.read_csv("artifacts/data_injection/data.csv")  # load your data
    
    data_processing = DataProcessing(config=data_preprocessing_config)
    data_processing.process(data)  # clean_genre and clean_text are called inside here

except Exception as e:
    raise e

[2026-05-28 13:35:31,152: INFO: common: YAML file loaded successfully from: config\config.yaml]
[2026-05-28 13:35:31,154: INFO: common: YAML file loaded successfully from: params.yaml]
[2026-05-28 13:35:31,155: INFO: common: created directory at artifacts]
[2026-05-28 13:35:31,157: INFO: common: created directory at artifacts/data_preprocessing]
